# Запуск MT-обучения в osiris: режим проверки и полноценный запуск

Архитектура S2+S3: эксперты без гейтов (concat `[E_task‖E_shared]`), общая CR-голова + STAR-башни CR по каналам, CTR-башня. Обучение joint, веса 1/1/1.

**Порядок работы**
1. Ячейки 1–3 один раз за сессию (импорты, Kerberos, параметры).
2. `RUN_MODE = "check"` → preflight → `DRY_RUN = False` → запуск → автоотчёт по логам.
3. Если отчёт без замечаний: `RUN_MODE = "full"`, `CONFIRM_FULL = True`, запуск.

Режим `check`: 1% файлов, 1 эпоха, 1 нода × 4 GPU, частое логирование и градиентный конфликт. Режим `full`: все файлы, 2 эпохи, 3 ноды × 4 GPU. Пресеты правятся в словаре `PRESETS`.

In [1]:
import os
import sys
import re
import glob
import json
import time
import shutil
import getpass
import subprocess

sys.path.insert(0, os.path.abspath("/opt/clients"))
import osiris

In [ ]:
PWD = ""

## 2. Kerberos / Hadoop (один раз за сессию)

In [ ]:
NFS = "/home/datalab/nfs"
PREPARE_ENV = True   # False, если ticket и конфиги уже подготовлены

if PREPARE_ENV:
    for name in ("hadoop", "nss", "krb5cc", "krb5.conf"):
        path = f"{NFS}/{name}"
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    shutil.copytree("/etc/hadoop/conf", f"{NFS}/hadoop", dirs_exist_ok=True)
    shutil.copy("/etc/krb5.conf", f"{NFS}/krb5.conf")
    subprocess.run(["kinit"], input=PWD + "\n", text=True, check=True)
    ticket = max(glob.glob("/tmp/krb5cc_*"), key=os.path.getmtime)
    shutil.copy(ticket, f"{NFS}/krb5cc")
    os.makedirs(f"{NFS}/nss", exist_ok=True)
    shutil.copy("/etc/passwd", f"{NFS}/nss/")
    print("Kerberos/Hadoop подготовлены, ticket:", ticket)

## 3. Параметры

In [ ]:
# ============ РЕЖИМ ============
RUN_MODE = "full"        # "check" — проверка; "full" — полноценное обучение
USE_MULTITASK = True      # True: MT-схема; False: прежний запуск --joint (для сравнения)
DRY_RUN = True            # True: только печать аргументов, задача не создаётся
CONFIRM_FULL = True      # защита от случайного полного запуска

PRESETS = {
    "check": dict(file_fraction=0.01, epochs=1, num_nodes=1, num_gpus=4, log_every=10, gc_every=50, suffix="_check"),
    "full":  dict(file_fraction=1.0,  epochs=2, num_nodes=4, num_gpus=4, log_every=50, gc_every=200, suffix=""),
}

# ============ ПУТИ ============
SCRIPT_PATH = "/home/datalab/nfs/Omni/TRAIN OMNI/5 MDMH/Train CR CTR EXPERTS CHNL/mdmh_cr_ctr_base_train.py"
OMNI_MAIN = "/home/datalab/nfs/Omni/omnichannel_main"          # PATH_OMNI_MAIN из .env
LOCAL_ROOT = "/home/datalab/nfs/Omni/"
CONFIG_PATH = "/home/datalab/nfs/Omni/configs_weights/omni_config.json"
DATASET_PATH = ""
MDL_BASE = "omni_ML_DCN_val_cr_ctr_mt_star_concat_small_head"

# ============ ДАННЫЕ ============
TRAIN_TABLE = "sdi_cr_ctr_dataset_val"
VAL_TABLE = ""
TST_TABLE = ""
OOT_TABLE = ""
DATA_FORMAT = "raw"
ROWS_CNT = 49_637_381          # строк в train
BATCH_SIZE = 128               # на один процесс
SEED = 42
UNK_EMB_PROB = 0.1
NUM_WORKERS = 2
TARGET_COLUMNS = "trgt_fin,trgt_ctr"
HEAD_TARGETS = '{"cr": "trgt_fin", "ctr": "trgt_ctr"}'
HEAD_WEIGHTS = '{"cr": 1.0, "ctr": 1.0}'

# ============ MT-АРХИТЕКТУРА ============
CHANNELS_PER_TASK = '{"cr": 4, "ctr": 1}'
EVALUATE_COLS = "eval_prod,eval_channel"
CHANNEL_NAMES = '{"cr": ["mass", "phygital", "premier", "sbol"]}'      # порядок = channel_ids
MT_CONFIG = {
    "channel_ids": {"cr": [0, 1, 2, 3]},                               # mass=0, phygital=1, premier=2, sbol=3
    "n_task_experts": {"cr": 1, "ctr": 1}, "n_shared_experts": 1,
    "expert_hidden": 1024, "expert_dim": 512, "expert_dropout": 0.2,
    "mix_type": "concat",                                               # S2: без гейтов, головы читают [E_task‖E_shared]
    "tower_type": "star", "star_hidden_dims": [256, 128],               # S3: канальные CR-башни STAR (W_s ⊙ (1+Δ_k))
    "tower_hidden": 256, "tower_dropout": 0.1,                          # tower_hidden — только для башни CTR
    "general_tasks": ["cr"], "general_tower_hidden_dims": [512, 256],   # общая CR-голова
}
MT_LRS = '{"experts": 1e-3, "towers": 1e-3, "star_delta": 1e-3}'
MT_WDS = '{"experts": 1e-2, "towers": 1e-2, "star_delta": 1e-1}'      # star_delta: тяга Δ_k -> 0 (канал -> общий путь)
NO_FIND_UNUSED = False

# ============ ПРЕЖНИЙ РЕЖИМ (USE_MULTITASK=False) ============
HEAD_KINDS = "cr,ctr"
CHANNEL_SPLIT = True

# ============ ЭМБЕДДИНГИ / ПОДМОДЕЛИ ============
EVK_WEIGHTS_PATH = "evk_weights.pt"
DLT_WEIGHTS_PATH = "dlt_weights.pt"
USE_EMB_FLAGS = """{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }"""

# ============ RESUME / ЗАМОРОЗКА ============
RESUME = None                  # путь к .pt
NO_LOAD_OPTIMIZER = False      # True, если у чекпоинта другое число param-групп
START_EPOCH = 0
ADAPTER_ONLY = False           # True: учить только mt.adapters (базу грузить через RESUME)
FREEZE_BACKBONE = False

# ============ РЕСУРСЫ ============
POOL = "b2c"

# ============ ПРОИЗВОДНЫЕ ============
assert RUN_MODE in PRESETS, f"RUN_MODE должен быть одним из {list(PRESETS)}"
P = PRESETS[RUN_MODE]
FILE_FRACTION, EPOCHS = P["file_fraction"], P["epochs"]
NUM_NODES, NUM_GPUS = P["num_nodes"], P["num_gpus"]
NUM_PROCESS = NUM_NODES * NUM_GPUS
MDL_NAME = MDL_BASE + P["suffix"]
ROWS_CNT_EFF = max(1, int(ROWS_CNT * FILE_FRACTION))      # расписание LR считается от реального числа строк
EXPECTED_TOTAL_PARAMS = 116_464_031                       # 113_405_383 (лог адаптерного запуска) - 19_872_303 (MT адаптера) + 22_930_951 (MT S2+S3)
EXPECTED_TRAINABLE_PARAMS = 33_933_727                    # 30_875_079 - 19_872_303 + 22_930_951, если EVK/DLT trainable не менялись
print(f"mode={RUN_MODE} multitask={USE_MULTITASK} model={MDL_NAME} nodes={NUM_NODES} procs={NUM_PROCESS} "
      f"rows_cnt={ROWS_CNT_EFF} epochs={EPOCHS} file_fraction={FILE_FRACTION}")

## 4. Preflight

Проверяет JSON-параметры, согласованность задач и каналов, валидирует `MT_CONFIG` через `MultiTaskConfig` (если доступен импорт), проверяет, что в `SCRIPT_PATH` и в `OMNI_MAIN` применены нужные правки (в том числе передача `mt_config` в модель, из-за пропуска которой падал первый запуск).

In [ ]:
def _fail(msg):
    raise AssertionError("PREFLIGHT: " + msg)

def preflight():
    parsed = {}
    for name in ("HEAD_TARGETS", "HEAD_WEIGHTS", "CHANNELS_PER_TASK", "CHANNEL_NAMES", "MT_LRS", "MT_WDS", "USE_EMB_FLAGS"):
        try:
            parsed[name] = json.loads(globals()[name])
        except Exception as e:
            _fail(f"{name} не парсится как JSON: {e}")
    cols = [c.strip() for c in TARGET_COLUMNS.split(",")]
    for task, col in parsed["HEAD_TARGETS"].items():
        if col not in cols:
            _fail(f"HEAD_TARGETS[{task}]={col} нет в TARGET_COLUMNS={cols}")
    if set(parsed["HEAD_WEIGHTS"]) != set(parsed["HEAD_TARGETS"]):
        _fail("ключи HEAD_WEIGHTS и HEAD_TARGETS расходятся")

    if not os.path.exists(SCRIPT_PATH):
        _fail(f"нет SCRIPT_PATH={SCRIPT_PATH}")
    text = open(SCRIPT_PATH, encoding="utf-8").read()

    if USE_MULTITASK:
        cps = parsed["CHANNELS_PER_TASK"]
        if set(cps) != set(parsed["HEAD_TARGETS"]):
            _fail("задачи CHANNELS_PER_TASK и HEAD_TARGETS расходятся")
        for task, k in cps.items():
            names = parsed["CHANNEL_NAMES"].get(task)
            if names is not None and len(names) != k:
                _fail(f"CHANNEL_NAMES[{task}] должен содержать {k} имён")
        groups = {"experts", "gates", "towers", "adapter", "star_delta"}
        bad = (set(parsed["MT_LRS"]) | set(parsed["MT_WDS"])) - groups
        if bad:
            _fail(f"MT_LRS/MT_WDS: неизвестные группы {bad}")
        if len([c for c in EVALUATE_COLS.split(",") if c.strip()]) < 2:
            _fail("EVALUATE_COLS должен содержать минимум 2 колонки: [prod, channel]")
        for tok in ("--multitask", "--channels-per-task", "--evaluate-cols", "--mt-lrs", "cycles_mt"):
            if tok not in text:
                _fail(f"в SCRIPT_PATH нет '{tok}': применены ли патчи к скрипту?")
        if not re.search(r"mt_config\s*=\s*mt_config", text):
            _fail("в SCRIPT_PATH модель создаётся без mt_config=mt_config (OmnichannelModel(..., mt_config=mt_config))")
        if MT_CONFIG.get("adapter_tasks") and "--train-adapter-only" not in text:
            print("WARN: в SCRIPT_PATH нет --train-adapter-only (stage5.patch не применён к скрипту)")
        if '(_ck.get("extra_data") or {})' not in text:
            print("WARN: resume не восстанавливает scheduler (правка F6 не применена)")

        src = f"{OMNI_MAIN}/src/omnichannel_model/omni"
        uf = f"{src}/utils.py"
        if MT_CONFIG.get("adapter_tasks") and os.path.exists(uf) and "mt.adapters." not in open(uf, encoding="utf-8").read():
            print("WARN: в utils.py нет группы mt.adapters: параметры адаптера попадут в группу experts")

        need = [("multitask.py", "class MultiTaskHead"), ("cycles_mt.py", "def train_multitask_osiris")]
        if MT_CONFIG.get("adapter_tasks"):
            need += [("multitask.py", "ChannelAdapter"), ("cycles_mt.py", "adapter_ids")]
        if MT_CONFIG.get("mix_type") == "concat" or MT_CONFIG.get("tower_type") == "star":
            need += [("multitask.py", "class StarTower"), ("multitask.py", "def expert_usage"),
                     ("cycles_mt.py", "expert_usage()"), ("cycles_mt.py", "for t in L.gate_util:"),
                     ("utils.py", "star_delta")]                       # без utils.py Δ_k молча попадут в towers с WD 1e-2
        for f, tok in need:
            fp = f"{src}/{f}"
            if not os.path.exists(fp):
                _fail(f"нет {fp} (проверьте OMNI_MAIN)")
            if tok not in open(fp, encoding="utf-8").read():
                _fail(f"{fp} не содержит '{tok}': S2+S3 не применён (python apply_s2_s3.py {OMNI_MAIN})")
        cfg_dict = dict(MT_CONFIG)
        cfg_dict.update(tasks=list(cps), channels_per_task=cps,
                        target_cols={h: cols.index(c) for h, c in parsed["HEAD_TARGETS"].items()})
        try:
            sys.path.insert(0, f"{OMNI_MAIN}/src")
            for _m in [m for m in sys.modules if m.startswith("omnichannel_model")]:
                del sys.modules[_m]                                    # иначе в ядре останется класс до патча
            from omnichannel_model.omni.multitask import MultiTaskConfig
            MultiTaskConfig.from_dict(cfg_dict)
            print("MT_CONFIG валиден (MultiTaskConfig)")
        except ImportError as e:
            print("WARN: MultiTaskConfig не импортируется в ноутбуке, валидация конфига пропущена:", e)
        except Exception as e:
            _fail(f"MT_CONFIG не проходит валидацию: {e}")
    if RESUME and not os.path.exists(RESUME):
        _fail(f"нет RESUME={RESUME}")
    if ADAPTER_ONLY and not RESUME:
        _fail("ADAPTER_ONLY=True без RESUME: у базы не будет обученных весов")
    print("PREFLIGHT: OK")
    return parsed

PARSED = preflight()

## 5. Аргументы и задача osiris

In [ ]:
def build_script_args():
    a = [
        "--num_machines", f"{NUM_NODES}",
        "--num_processes", f"{NUM_PROCESS}",
        "--main_process_ip", "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port", "$(MASTER_PORT)",
        "--machine_rank", "$(RANK)",
        f"{SCRIPT_PATH}",
        "--local-root", str(LOCAL_ROOT),
        "--config-path", str(CONFIG_PATH),
        "--mdl-name", str(MDL_NAME),
        "--data-format", str(DATA_FORMAT),
        "--train-table", str(TRAIN_TABLE),
        "--rows-cnt", str(ROWS_CNT_EFF),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--num-workers", str(NUM_WORKERS),
        "--file-fraction", str(FILE_FRACTION),
        "--unk-emb-prob", str(UNK_EMB_PROB),
        "--target-columns", str(TARGET_COLUMNS),
        "--head-targets", HEAD_TARGETS,
        "--head-weights", HEAD_WEIGHTS,
        "--evk-weights-path", EVK_WEIGHTS_PATH,
        "--dlt-weights-path", DLT_WEIGHTS_PATH,
        "--use-emb-flags", str(USE_EMB_FLAGS),
        "--start-epoch", str(START_EPOCH),
    ]
    if DATASET_PATH:
        a += ["--dataset-path", DATASET_PATH]
    for flag, val in (("--val-table", VAL_TABLE), ("--tst-table", TST_TABLE), ("--oot-table", OOT_TABLE)):
        if val:
            a += [flag, val]
    if USE_MULTITASK:
        a += ["--multitask",
              "--channels-per-task", CHANNELS_PER_TASK,
              "--mt-config", json.dumps(MT_CONFIG),
              "--evaluate-cols", EVALUATE_COLS,
              "--channel-names", CHANNEL_NAMES,
              "--mt-lrs", MT_LRS, "--mt-wds", MT_WDS,
              "--log-every", str(P["log_every"]),
              "--gc-every", str(P["gc_every"])]
        if NO_FIND_UNUSED:
            a += ["--no-find-unused"]
        if ADAPTER_ONLY:
            a += ["--train-adapter-only"]
    else:
        a += ["--head-kinds", str(HEAD_KINDS), "--joint", "--log-every", str(P["log_every"])]
        if CHANNEL_SPLIT:
            a += ["--channel-split"]
    if RESUME:
        a += ["--resume-from", str(RESUME)]
        if NO_LOAD_OPTIMIZER:
            a += ["--no-load-optimizer"]
    if FREEZE_BACKBONE:
        a += ["--freeze-backbone"]
    return a


def create_job(job_name, num_gpus):
    return osiris.create(
        name=f"sc-{job_name}",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
        restart=False,
        pool=POOL,
        script=SCRIPT_PATH,
        command=["accelerate", "launch"],
        args=build_script_args(),
        envs={
            "PYTHONPATH": "",
            "OMP_NUM_THREADS": "30",
            "NCCL_DEBUG": "INFO",
            "HADOOP_CONF_DIR": "/home/datalab/nfs/hadoop",
            "KRB5_CONFIG": "/home/datalab/nfs/krb5.conf",
            "KRB5CCNAME": "/home/datalab/nfs/krb5cc",
            "ARROW_LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "HADOOP_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LD_PRELOAD": "/usr/lib64/libnss_wrapper.so",
            "NSS_WRAPPER_PASSWD": "/home/datalab/nfs/nss/passwd",
            "NSS_WRAPPER_GROUP": "/etc/group",
            "CLASSPATH": f":/home/datalab/nfs/hadoop:" + os.environ["CLASSPATH"].split(":", 2)[-1],
            "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
            "NCCL_SOCKET_FAMILY": "AF_INET",
            "GLOO_SOCKET_FAMILY": "AF_INET",
            "NCCL_SOCKET_IFNAME": "eth0",
            "GLOO_SOCKET_IFNAME": "eth0",
            "NCCL_IB_DISABLE": "1",
            "NCCL_P2P_DISABLE": "1",
            "NCCL_SHM_DISABLE": "1",
        },
        num_nodes=NUM_NODES,
        num_gpus=num_gpus,
        type="pytorchjob",
    )

print(" ".join(f"'{x}'" if (" " in x or x[:1] in "{[") else x for x in build_script_args()))

## 6. Запуск

`DRY_RUN = True` ничего не создаёт. Для `RUN_MODE = "full"` нужно ещё `CONFIRM_FULL = True`. В логе мастера сразу после старта ищите строки `[MODEL] Total params` и `[MT] первый батч`.

In [8]:
DRY_RUN = False

In [ ]:
JOB_NAME = f"train-cr-ctr-mt-{RUN_MODE}" if USE_MULTITASK else f"train-cr-ctr-{RUN_MODE}"

if DRY_RUN:
    print("DRY_RUN: задача не создавалась")
else:
    if RUN_MODE == "full" and not CONFIRM_FULL:
        raise RuntimeError("Для полноценного запуска установите CONFIRM_FULL = True (после успешного режима check)")
    PARSED = preflight()
    job = create_job(JOB_NAME, NUM_GPUS)
    print(job)
    job_name = job["job"]

## 7. Мониторинг и автоотчёт по логам

In [ ]:
def _find_job(name):
    for j in osiris.list()["jobs"]:
        if j["job"] == name:
            return j
    return None


def check_report(text, expected_total=EXPECTED_TOTAL_PARAMS):
    """Автоматическая проверка логов мастера. Возвращает список замечаний (пусто = всё в порядке)."""
    issues = []
    m = re.search(r"\[MODEL\] Total params: ([\d,]+)\s+Trainable: ([\d,]+)", text)
    if not m:
        issues.append("нет строки [MODEL] Total params")
    else:
        total, trainable = (int(x.replace(",", "")) for x in m.groups())
        print(f"параметры: всего {total:,}, обучаемых {trainable:,}")
        if USE_MULTITASK and total != expected_total:
            issues.append(f"всего параметров {total:,}, ожидалось {expected_total:,} (если конфигурация менялась, это нормально)")
        if USE_MULTITASK and trainable != EXPECTED_TRAINABLE_PARAMS:
            issues.append(f"обучаемых {trainable:,}, ожидалось {EXPECTED_TRAINABLE_PARAMS:,} (проверьте заморозку EVK/DLT и mt)")
    mc_cfg = re.search(r"\[MT\] multitask-режим[^\n]*", text)
    if USE_MULTITASK and mc_cfg:
        for tok in (f"mix_type='{MT_CONFIG.get('mix_type', 'gate')}'", f"tower_type='{MT_CONFIG['tower_type']}'"):
            if tok not in mc_cfg.group(0):
                issues.append(f"в [MT]-конфиге задачи нет {tok}: задача собрана не с тем кодом/конфигом")
    fb = re.search(r"\[MT\] первый батч: (.*)", text)
    if USE_MULTITASK:
        if not fb:
            issues.append("нет строки [MT] первый батч")
        else:
            line = fb.group(1)
            print("первый батч:", line)
            mc = re.search(r"ctr: labeled=(\d+)/(\d+)", line)
            if mc and not (15 <= int(mc.group(1)) <= 70):
                issues.append(f"в первом батче {mc.group(1)} CTR-строк из {mc.group(2)}: ожидается около 38 при перемешанных данных")
            if "неизвестные каналы" in line:
                issues.append("в данных есть каналы вне channel_ids")
    if "Traceback" in text:
        issues.append(f"в логе есть Traceback ({text.count('Traceback')})")
    if "non-finite loss" in text:
        issues.append("встретился нечисловой лосс")
    if "Error processing file" in text:
        issues.append("ошибки чтения файлов (Error processing file): часть данных могла быть пропущена")
    if "[CKPT] Saved" not in text:
        issues.append("нет строки [CKPT] Saved: чекпоинт мог не сохраниться")
    last = None
    for last in re.finditer(r"Epoch \d+, Batch \d+, Loss: [^\r\n]*", text):
        pass
    if last:
        print("последняя строка прогресса:", last.group(0)[:200])
        if USE_MULTITASK:
            for key in loss_keys():
                if f"{key}:" not in last.group(0):
                    issues.append(f"в строке прогресса нет лосса '{key}'")
    else:
        issues.append("нет строк прогресса (обучение не стартовало?)")
    return issues


def loss_keys():
    keys = (["cr"] if MT_CONFIG.get("tower_type") != "none" else []) + ["ctr"]
    keys += ["cr_general"] if "cr" in MT_CONFIG.get("general_tasks", []) else []
    keys += ["cr_adj"] if MT_CONFIG.get("adapter_tasks") else []
    return keys


def metrics_commands():
    d = os.path.dirname(SCRIPT_PATH)
    wdir = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/weights"
    chk = f"{MDL_NAME}_ep{EPOCHS - 1}_weights.pt"
    for head in loss_keys():
        print(f"accelerate launch {d}/base_ddp_metrics.py --config-path {CONFIG_PATH} --mdl-name {MDL_NAME} "
              f"--chkpt-name {chk} --multitask --mt-config {wdir}/{MDL_NAME}_mt_config.json "
              f"--target-columns {TARGET_COLUMNS} --head-name {head} --eval-splits val --val-table <ТАБЛИЦА_VAL> "
              f"--use-emb-flags '{' '.join(USE_EMB_FLAGS.split())}'")


if not DRY_RUN:
    my_job = _find_job(job_name)
    while my_job is None or my_job["state"] not in ("Succeeded", "Failed"):
        my_job = _find_job(job_name)
        if my_job is not None:
            print(my_job["state"], end="\r")
        time.sleep(5)
    print("Состояние:", my_job["state"])
    try:
        out = osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=20000)
        text = out if isinstance(out, str) else (str(out) if out is not None else "")
    except Exception as e:
        text = ""
        print("не удалось получить логи автоматически:", e)
    if text:
        issues = check_report(text)
        print("\nЗамечания:" if issues else "\nЗамечаний нет")
        for i in issues:
            print(" -", i)
    else:
        print("Логи не получены строкой: посмотрите их в следующей ячейке и проверьте вручную")
    if my_job["state"] == "Succeeded":
        print("\nКоманды для расчёта метрик (подставьте таблицу val):")
        metrics_commands()

In [13]:
log_name = '5_log.pkl'

In [ ]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
import pickle

raw = osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=2_000_000)  # как в ноутбуке
pickle.dump(raw, open(log_name, 'wb'))
rows = {}
for line in raw['pods'][0]['logs']:#[33_000:37_000] :
    m = re.search(r"Epoch (\d+), Batch (\d+), Loss: ([0-9.]+).*?LR: ([0-9.]+e[+-]\d+)", line)
    if m:
        rows[(int(m[1]), int(m[2]))] = (float(m[3]), float(m[4]))

df = (pd.DataFrame.from_dict(rows, orient="index", columns=["loss_cum", "lr"])
        .rename_axis(["epoch"]).reset_index())
df['batch'] = df.epoch.apply(lambda x: x[1])
df['epoch'] = df.epoch.apply(lambda x: x[0])
df = df[df["batch"] > 1].copy()                       # отбрасываем первую точку (loss=0.0000)

df["b"] = df["batch"] - 1                             # batch_idx
df["S"] = df["loss_cum"] * (df["b"] + 1)              # сумма лоссов предыдущих батчей
df["loss_win"] = df.groupby("epoch")["S"].diff() / df.groupby("epoch")["b"].diff()

last = df.groupby("epoch")["batch"].max()
offset = last.cumsum().shift(fill_value=0)
df["step"] = df["batch"] + df["epoch"].map(offset)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df["step"], df["loss_cum"], lw=1, alpha=0.5, label="накопленное среднее (как в логе)")
#ax.plot(df["step"], df["loss_win"], lw=1.5, label="средний лосс между точками лога")
for ep, off in offset.items():
    if ep > df["epoch"].min():
        ax.axvline(off, color="grey", ls="--", lw=0.8)
ax.set_xlabel("Шаг на одном ранге (батчи)")
ax.set_ylabel("Loss")
ax.grid(alpha=0.3)

ax2 = ax.twinx()                                      # LR на второй оси
ax2.plot(df["step"], df["lr"], color="red", lw=1, alpha=0.6, label="LR")
ax2.set_ylabel("LR")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
# логи мастера вручную (ищите [MODEL], [MT], Warning, Traceback)
osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=300)

In [ ]:
# удаление задачи
osiris.delete(osiris.list())